# 1 · Python Foundations
*Intro to Python for Scientists & Public Health Professionals*

**Who this is for.** This course assumes you already program in *something* (R, SAS, MATLAB, Stata, SQL, Java...). We will not dwell on what a loop or a variable is — instead we focus on how Python does these things and where it differs from what you already know.

### By the end of this notebook you can
- Work with Python's core types: numbers, strings, booleans
- Choose between the four built-in collections (list, tuple, set, dict) and know which are mutable
- Write idiomatic control flow and comprehensions
- Avoid the aliasing/mutability traps that bite people coming from other languages
- Read a traceback and handle errors with `try`/`except`

### Agenda
1. Variables & numeric types
2. Strings & f-strings
3. Booleans & truthiness
4. Collections: list · tuple · set · dict
5. Control flow
6. Comprehensions
7. Errors & exceptions

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## Coming from another language? Python in 60 seconds

| Thing | What to know |
|---|---|
| **Indexing** | Starts at **0**. `x[-1]` is the last item. |
| **Indentation** | Whitespace *is* syntax — it defines blocks. No `{}` or `end`. |
| **Typing** | Dynamic. You never declare a type; you just assign. |
| **Everything is an object** | Numbers, functions, even types — all have methods/attributes. |
| **Mutability matters** | Some objects can be changed in place (lists, dicts); some cannot (tuples, strings). This has real consequences — see §4. |
| **Truthiness** | Empty containers, `0`, `''`, and `None` are "falsy". You rarely compare to `True`. |

Keep these in mind; most surprises trace back to one of them.

## 1. Variables and numeric types

No declarations, no type keywords — assign and go. The three numeric types are `int`, `float`, and `complex`.

In [ ]:
# Python infers the type from the value
doses_shipped = 1000        # int
coverage_rate = 0.873       # float

print(doses_shipped, type(doses_shipped))
print(coverage_rate, type(coverage_rate))

In [ ]:
# Complex numbers are first-class (handy in signal processing, FFTs, etc.)
# The imaginary unit is written j, not i
z = 3 + 4j
print(z.real, z.imag, abs(z))   # 3.0  4.0  5.0

> **Note:** scientific notation such as `2.3e4` is just a `float` (= 23000.0), *not* a complex number — a common point of confusion.

In [ ]:
# Operators worth a reminder
print(17 // 5)   # floor division -> 3
print(17 % 5)    # modulus (remainder) -> 2
print(2 ** 10)   # exponent -> 1024

## 2. Strings & f-strings

Use **f-strings** for everything — they are the modern standard and far more readable than `%` or `.format()`.

In [ ]:
site = "Riverside Clinic"
administered = 947
eligible = 1085
rate = administered / eligible

# Format specs go after a colon inside the braces
print(f"{site}: {administered} of {eligible} doses given ({rate:.1%})")
print(f"Remaining: {eligible - administered:,} people")   # thousands separator

> The mini-language for format specs (`.1%`, `,`, padding, alignment) is worth bookmarking: [Format Specification Mini-Language](https://docs.python.org/3/library/string.html#format-spec).

In [ ]:
# A few string methods you'll reach for when cleaning data
raw = "  Diabetes_Type_2  "
clean = raw.strip().lower().replace("_", " ")
print(repr(clean))        # 'diabetes type 2'
print(clean.split())      # ['diabetes', 'type', '2']

## 3. Booleans & truthiness

A boolean is `True` or `False`. Comparisons and the operators `and` / `or` / `not` return booleans.

In [ ]:
a, b = 100, 95
print(a > b)                  # True
print((a > b) and (b > 50))   # True
print((a < b) or (b == 95))   # True

# Write `if condition:` — never `if condition == True:`

In [ ]:
# Falsy values: empty containers, 0, '', and None
for value in [0, "", [], {}, None, 42, "ok", [1]]:
    print(f"{repr(value):>6}  ->  {bool(value)}")

In [ ]:
# This is why the idiomatic "is it empty?" check is just `if x:`
records = []
if records:
    print("processing records")
else:
    print("no records yet")

## 4. Collections: list · tuple · set · dict

| Type | Syntax | Ordered? | Mutable? | Duplicates? |
|---|---|---|---|---|
| **list** | `[...]` | yes | **yes** | yes |
| **tuple** | `(...)` | yes | no | yes |
| **set** | `{...}` | no | **yes** | no |
| **dict** | `{k: v}` | yes (3.7+) | **yes** | unique keys |

### Lists

In [ ]:
sites = ["Riverside", "Hilltop", "Downtown", "Eastgate"]
print(sites[0], sites[-1])   # first, last
print(sites[1:3])            # slice -> ['Hilltop', 'Downtown']
print(sites[::-1])           # reversed copy

In [ ]:
sites = ["Riverside", "Hilltop"]
sites.append("Downtown")               # add one
sites.extend(["Eastgate", "Midtown"])  # add several
sites.insert(0, "Central")             # at a position
print(sites)

> **Gotcha — lists are references, not values.** Assignment does *not* copy. This trips up almost everyone arriving from a value-semantics language.

In [ ]:
a = [1, 2, 3]
b = a              # b is NOT a copy — it's the same list
b.append(99)
print(a)           # [1, 2, 3, 99]  <- 'a' changed too!

c = a.copy()       # make an independent copy
c.append(0)
print(a)           # unchanged this time

### Tuples — immutable, great for fixed records and unpacking

In [ ]:
point = (39.95, -75.16)       # (lat, lon) — can't be changed
lat, lon = point              # unpacking
print(lat, lon)

lat, lon = lon, lat           # swap with no temp variable
print(lat, lon)

### Sets — unordered, unique, and **mutable**

A set holds **unique** items in **no particular order**, and it **can** be modified after creation (`.add`, `.discard`). The immutable cousin is `frozenset`. Sets shine for de-duplication and membership math.

In [ ]:
visits = {"A", "B", "A", "C", "B"}
print(visits)            # duplicates collapsed
visits.add("D")
visits.discard("Z")      # no error if missing (.remove would raise KeyError)
print(visits)

#### Exercise 1 — Set operations *(5 min)*

Two collections of locations are below. Using set operations, find:
1. Sites that received a delivery **and** run a clinic (intersection)
2. Sites in exactly **one** of the two sets (symmetric difference)

Docs: https://docs.python.org/3/library/stdtypes.html#set

In [ ]:
deliveries = {"Boston", "Monticello", "Chicago", "Atlanta", "Dickey", "Douglas", "Zenda", "Springfield"}
clinics    = {"Boston", "Chicago", "Atlanta", "Zenda", "Springfield"}

# Your work here


<details>
<summary>Solution</summary>

```python
print(deliveries & clinics)   # intersection
print(deliveries ^ clinics)   # symmetric difference (in exactly one)

# equivalent method form:
print(deliveries.intersection(clinics))
print(deliveries.symmetric_difference(clinics))
```

**Why this works.** Sets support mathematical operators directly: `&` keeps items present in *both* sets, `^` keeps items in *one set but not both*, `|` is union, and `-` is difference. The operator and method forms are interchangeable, but operators only work when both sides are already sets, whereas `.intersection(...)` accepts any iterable — useful when one side is a list.

</details>

### Dictionaries — key:value pairs

In [ ]:
patient = {"id": 4172, "age": 58, "a1c": 7.4, "site": "Riverside"}

print(patient["age"])                      # direct access
print(patient.get("bmi", "not recorded"))  # safe access with a default

patient["bmi"] = 31.2                       # add a key
patient.update({"a1c": 7.1})                # update a value

for key, value in patient.items():
    print(f"{key}: {value}")

#### Exercise 2 — Build and query a record *(10 min)*

1. Create a dict describing a clinic with at least 5 key:value pairs.
2. Safely read a key that might be missing, returning `"unknown"` if absent.
3. Add one new key:value pair.
4. Print all values as a list.

Docs: https://docs.python.org/3/library/stdtypes.html#dict

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
clinic = {
    "name": "Riverside",
    "county": "Hudson",
    "beds": 42,
    "has_pharmacy": True,
    "weekly_visits": 610,
}

print(clinic.get("director", "unknown"))   # 2. safe read of a missing key
clinic["zip"] = "07030"                     # 3. add a pair
print(list(clinic.values()))                # 4. all values
```

**Why this works.** `clinic["director"]` would raise `KeyError` because that key doesn't exist — `.get("director", "unknown")` returns the fallback instead, which is the safe pattern whenever a key may be absent. Assigning to a new key (`clinic["zip"] = ...`) adds it in place. `.values()` returns a *view* (not a list), so we wrap it in `list(...)` to get a plain list.

</details>

## 5. Control flow

`if`/`elif`/`else`, `for`, and `while`. Note Python has no `switch` — chained `elif` does that job.

In [ ]:
a1c = 7.4
if a1c >= 9.0:
    risk = "high"
elif a1c >= 7.0:
    risk = "elevated"
else:
    risk = "controlled"
print(risk)

**Looping idioms.** When you need the index, use `enumerate` — not `range(len(...))`. When you need to walk two sequences together, use `zip`.

In [ ]:
sites = ["Riverside", "Hilltop", "Downtown"]

for i, site in enumerate(sites, start=1):   # idiomatic indexed loop
    print(i, site)

In [ ]:
counts = [947, 612, 388]
for site, count in zip(sites, counts):       # parallel iteration
    print(f"{site}: {count}")

In [ ]:
# Walk a dictionary
patient = {"id": 4172, "age": 58, "a1c": 7.1}
for key, value in patient.items():
    print(key, "=", value)

In [ ]:
# while loop + range
for n in range(5, 0, -1):   # 5,4,3,2,1
    print(n, end=" ")
print("liftoff")

## 6. Comprehensions

Compact, fast ways to build a collection from another. Available for lists, dicts, and sets.

> More patterns and the formal syntax: [List, set & dict comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions).

In [ ]:
counts = [947, 612, 388, 0, 155]

nonzero = [c for c in counts if c > 0]   # filter
doubled = [c * 2 for c in counts]        # transform
print(nonzero)
print(doubled)

In [ ]:
sites  = ["Riverside", "Hilltop", "Downtown"]
counts = [947, 612, 388]

by_site      = {s: c for s, c in zip(sites, counts)}   # dict comprehension
first_letters = {s[0] for s in sites}                  # set comprehension
print(by_site)
print(first_letters)

#### Exercise 3 — Normalize a list *(5 min)*

Given the list below, use a **list comprehension** to produce a new list with every name in UPPER CASE.

`states = ["Nebraska", "Oklahoma", "Illinois", "Georgia", "Ohio"]`

In [ ]:
states = ["Nebraska", "Oklahoma", "Illinois", "Georgia", "Ohio"]

# Your work here


<details>
<summary>Solution</summary>

```python
upper = [s.upper() for s in states]
print(upper)
```

**Why this works.** Read it left-to-right as "give me `s.upper()` for each `s` in `states`." The comprehension builds and returns a brand-new list, leaving `states` untouched — so it's both more concise and safer than mutating the original in a loop.

</details>

## 7. Errors & exceptions

You will hit errors constantly — being fluent at *reading* them is a real skill.

**Anatomy of a traceback:** read it **bottom-up**. The last line names the **exception type** and a **message**; the lines above show *where* it happened (the call stack). For a `KeyError: 'a1c'`, the type is `KeyError` and the missing key is `'a1c'`.

In [ ]:
# This line would raise KeyError: 'a1c'.
# Uncomment it during class to see a real traceback, then re-comment so the notebook runs clean.
patient = {"id": 4172, "age": 58}

# patient["a1c"]

Handle anticipated errors with `try` / `except`. `else` runs if no error; `finally` always runs.

In [ ]:
patient = {"id": 4172, "age": 58}

try:
    value = patient["a1c"]
except KeyError:
    value = None
    print("a1c not recorded for this patient")
else:
    print("found:", value)
finally:
    print("done checking")

**Exceptions you'll see most often**

| Exception | Typical cause |
|---|---|
| `KeyError` | dict key doesn't exist |
| `IndexError` | list index out of range |
| `ValueError` | right type, wrong value (e.g. `int("abc")`) |
| `TypeError` | wrong type (e.g. `"3" + 5`) |
| `FileNotFoundError` | path doesn't exist |
| `KeyError` / `AttributeError` | very common once we hit pandas |

A good habit: catch the *specific* exception you expect, not a bare `except:` that hides real bugs.

> The full hierarchy of built-in exceptions: [Built-in Exceptions](https://docs.python.org/3/library/exceptions.html).

## Wrap-up

You now have Python's core types, the four collections (and which are mutable), idiomatic control flow and comprehensions, the aliasing trap, and basic error handling.

**Next:** Functions — packaging this logic into reusable, testable pieces.